A small example case of machine learning.

In [1]:
from pyspark.sql import SparkSession

import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

In [3]:
spark = SparkSession.builder.appName('Practise').getOrCreate()

In [9]:
training = spark.read.csv('AustraliaWeather\Weather_Data.csv',header=True,inferSchema=True) 

In [10]:
training

DataFrame[Date: string, MinTemp: double, MaxTemp: double, Rainfall: double, Evaporation: double, Sunshine: double, WindGustDir: string, WindGustSpeed: int, WindDir9am: string, WindDir3pm: string, WindSpeed9am: int, WindSpeed3pm: int, Humidity9am: int, Humidity3pm: int, Pressure9am: double, Pressure3pm: double, Cloud9am: int, Cloud3pm: int, Temp9am: double, Temp3pm: double, RainToday: string, RainTomorrow: string]

In [11]:
training.show()

+---------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+
|     Date|MinTemp|MaxTemp|Rainfall|Evaporation|Sunshine|WindGustDir|WindGustSpeed|WindDir9am|WindDir3pm|WindSpeed9am|WindSpeed3pm|Humidity9am|Humidity3pm|Pressure9am|Pressure3pm|Cloud9am|Cloud3pm|Temp9am|Temp3pm|RainToday|RainTomorrow|
+---------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+
| 2/1/2008|   19.5|   22.4|    15.6|        6.2|     0.0|          W|           41|         S|       SSW|          17|          20|         92|         84|     1017.6|     1017.4|       8|       8|   20.7|   20.9|      Yes|         Yes|
| 2/2/2008|   19.5|   25.6|     6.0|        3.4|    

In [12]:
training.summary().show()

+-------+--------+------------------+-----------------+-----------------+------------------+------------------+-----------+------------------+----------+----------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+---------+------------+
|summary|    Date|           MinTemp|          MaxTemp|         Rainfall|       Evaporation|          Sunshine|WindGustDir|     WindGustSpeed|WindDir9am|WindDir3pm|      WindSpeed9am|      WindSpeed3pm|       Humidity9am|       Humidity3pm|       Pressure9am|       Pressure3pm|          Cloud9am|         Cloud3pm|           Temp9am|          Temp3pm|RainToday|RainTomorrow|
+-------+--------+------------------+-----------------+-----------------+------------------+------------------+-----------+------------------+----------+----------+------------------+------------------+------------------+------------------+--------

Trying to predict rainfall based on evaporation and sunshine

In [13]:
training.printSchema()

root
 |-- Date: string (nullable = true)
 |-- MinTemp: double (nullable = true)
 |-- MaxTemp: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Evaporation: double (nullable = true)
 |-- Sunshine: double (nullable = true)
 |-- WindGustDir: string (nullable = true)
 |-- WindGustSpeed: integer (nullable = true)
 |-- WindDir9am: string (nullable = true)
 |-- WindDir3pm: string (nullable = true)
 |-- WindSpeed9am: integer (nullable = true)
 |-- WindSpeed3pm: integer (nullable = true)
 |-- Humidity9am: integer (nullable = true)
 |-- Humidity3pm: integer (nullable = true)
 |-- Pressure9am: double (nullable = true)
 |-- Pressure3pm: double (nullable = true)
 |-- Cloud9am: integer (nullable = true)
 |-- Cloud3pm: integer (nullable = true)
 |-- Temp9am: double (nullable = true)
 |-- Temp3pm: double (nullable = true)
 |-- RainToday: string (nullable = true)
 |-- RainTomorrow: string (nullable = true)



In [14]:
training.columns

['Date',
 'MinTemp',
 'MaxTemp',
 'Rainfall',
 'Evaporation',
 'Sunshine',
 'WindGustDir',
 'WindGustSpeed',
 'WindDir9am',
 'WindDir3pm',
 'WindSpeed9am',
 'WindSpeed3pm',
 'Humidity9am',
 'Humidity3pm',
 'Pressure9am',
 'Pressure3pm',
 'Cloud9am',
 'Cloud3pm',
 'Temp9am',
 'Temp3pm',
 'RainToday',
 'RainTomorrow']

In [16]:
from pyspark.ml.feature import VectorAssembler  

#grouping evaporation and sunshine together ---> new feature ---> independent feature
featureAssembler = VectorAssembler(inputCols=["Evaporation","Sunshine"], outputCol="Independent Feature")

In [17]:
output = featureAssembler.transform(training)

In [18]:
output.show()

+---------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+-------------------+
|     Date|MinTemp|MaxTemp|Rainfall|Evaporation|Sunshine|WindGustDir|WindGustSpeed|WindDir9am|WindDir3pm|WindSpeed9am|WindSpeed3pm|Humidity9am|Humidity3pm|Pressure9am|Pressure3pm|Cloud9am|Cloud3pm|Temp9am|Temp3pm|RainToday|RainTomorrow|Independent Feature|
+---------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+-------------------+
| 2/1/2008|   19.5|   22.4|    15.6|        6.2|     0.0|          W|           41|         S|       SSW|          17|          20|         92|         84|     1017.6|     1017.4|       8|       8|   20.7|   20.9|      Yes|      

In [19]:
output.columns 

['Date',
 'MinTemp',
 'MaxTemp',
 'Rainfall',
 'Evaporation',
 'Sunshine',
 'WindGustDir',
 'WindGustSpeed',
 'WindDir9am',
 'WindDir3pm',
 'WindSpeed9am',
 'WindSpeed3pm',
 'Humidity9am',
 'Humidity3pm',
 'Pressure9am',
 'Pressure3pm',
 'Cloud9am',
 'Cloud3pm',
 'Temp9am',
 'Temp3pm',
 'RainToday',
 'RainTomorrow',
 'Independent Feature']

In [20]:
finalised_data = output.select("Independent Feature", "Rainfall")

In [21]:
finalised_data.show()

+-------------------+--------+
|Independent Feature|Rainfall|
+-------------------+--------+
|          [6.2,0.0]|    15.6|
|          [3.4,2.7]|     6.0|
|          [2.4,0.1]|     6.6|
|          [2.2,0.0]|    18.8|
|          [4.8,0.0]|    77.4|
|          [2.6,8.6]|     1.6|
|          [5.2,5.2]|     6.2|
|          [5.8,2.1]|    27.6|
|          [4.8,3.0]|    12.6|
|         [4.4,10.1]|     8.8|
|          [6.4,8.0]|     0.0|
|          [6.8,6.7]|     0.0|
|          [7.0,3.3]|    14.4|
|          [3.2,8.7]|     3.0|
|          [6.2,8.5]|     0.0|
|          [6.2,8.8]|     0.0|
|          [7.6,3.2]|     0.0|
|          [4.2,4.5]|     0.0|
|          [5.2,7.5]|     0.0|
|         [4.6,11.1]|     0.0|
+-------------------+--------+
only showing top 20 rows


In [23]:
from pyspark.ml.regression import LinearRegression

#train test split
train_data, test_data = finalised_data.randomSplit([0.75,0.25])
regressor = LinearRegression(featuresCol='Independent Feature', labelCol='Rainfall')
regressor = regressor.fit(train_data)

In [24]:
regressor.coefficients

DenseVector([-0.2377, -0.8171])

In [25]:
regressor.intercept

10.548974290043542

In [26]:
#prediction
pred_result = regressor.evaluate(test_data)

In [27]:
pred_result.predictions.show()

+-------------------+--------+------------------+
|Independent Feature|Rainfall|        prediction|
+-------------------+--------+------------------+
|          [0.0,6.0]|     4.4| 5.646572387166387|
|          [0.2,6.1]|     7.8| 5.517323623570524|
|          [0.2,7.4]|     0.0| 4.455136544613807|
|          [0.2,8.5]|     0.8| 3.556362862419662|
|          [0.4,0.0]|     4.0|10.453890159614389|
|          [0.4,0.2]|    56.2|10.290476762851817|
|          [0.4,3.5]|     4.2|7.5941557162693805|
|          [0.4,6.3]|     0.2| 5.306368161593376|
|          [0.4,6.4]|     7.8| 5.224661463212089|
|          [0.4,7.7]|     3.6| 4.162474384255372|
|          [0.4,8.3]|     0.6|3.6722341939676566|
|          [0.4,9.0]|    12.2|3.1002873052986555|
|          [0.6,0.0]|     0.0|10.406348094399812|
|          [0.6,0.6]|     3.6| 9.916107904112096|
|          [0.6,1.9]|    31.8|  8.85392082515538|
|          [0.6,7.7]|     3.0| 4.114932319040794|
|          [0.6,8.7]|     1.0| 3.297865335227936|


In [28]:
pred_result.meanAbsoluteError, pred_result.meanSquaredError

(4.295444237745347, 61.57393619691763)